In [35]:
%%writefile /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/preprocessing/feature_utils.py

import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


PRICE_MAP = {
    "50-100": 0,
    "100-150": 1,
    "150-200": 2,
    "200-250": 3,
}

AGE_GROUP_MAP = {
    "18-25": 1,
    "26-35": 2,
    "36-45": 3,
    "46-55": 4,
    "56-70": 5,
}

INCOME_MAP = {
    "Not Reported": 0,
    "<10L": 1,
    "10L - 15L": 2,
    "16L - 25L": 3,
    "26L - 35L": 4,
    "> 35L": 5,
}

HEALTH_MAP = {
    "Low (Not very concerned)": 1,
    "Medium (Moderately health-conscious)": 2,
    "High (Very health-conscious)": 3,
}

FREQ_MAP = {
    "0-2 times": 1,
    "3-4 times": 2,
    "5-7 times": 3,
}

SIZE_MAP = {
    "Small (250 ml)": 1,
    "Medium (500 ml)": 2,
    "Large (1 L)": 3,
}

CF_MAP = FREQ_MAP

AB_MAP = {
    "0 to 1": 1,
    "2 to 4": 2,
    "above 4": 3,
}

ZONE_MAP = {
    "Rural": 1,
    "Semi-Urban": 2,
    "Urban": 3,
    "Metro": 4,
}

IMPUTE_COLS = [
    "consume_frequency(weekly)",
    "purchase_channel",
]


def _business_features(df):
    df = df.copy()

    # Idempotent deterministic cleaning
    df["zone"] = df["zone"].replace({
        "urbna": "Urban",
        "Metor": "Metro",
    })

    df["current_brand"] = df["current_brand"].replace({
        "newcomer": "Newcomer",
        "Establishd": "Established",
    })

    df["income_levels"] = (
        df["income_levels"]
        .fillna("Not Reported")
        .str.strip()
    )

    # Recompute from raw age so serving uses same rule
    df["age_group"] = pd.cut(
        df["age"],
        bins=[17, 25, 35, 45, 55, 70],
        labels=[
            "18-25",
            "26-35",
            "36-45",
            "46-55",
            "56-70",
        ],
        include_lowest=True
    ).astype(str)

    df["zone_num"] = df["zone"].map(ZONE_MAP)
    df["income_num"] = df["income_levels"].map(INCOME_MAP)

    df["zas_score"] = (
        df["zone_num"] * df["income_num"]
    )

    df["bsi"] = (
        (df["current_brand"] != "Established")
        &
        (
            df["reasons_for_choosing_brands"]
            .isin(["Price", "Quality"])
        )
    ).astype(int)

    df["cf_num"] = (
        df["consume_frequency(weekly)"].map(CF_MAP)
    )

    df["ab_num"] = (
        df["awareness_of_other_brands"].map(AB_MAP)
    )

    df["cf_ab_score"] = (
        df["cf_num"]
        /
        (df["cf_num"] + df["ab_num"])
    ).round(2)

    # Fixed ordinal mappings
    df["age_group"] = df["age_group"].map(AGE_GROUP_MAP)
    df["income_levels"] = df["income_levels"].map(INCOME_MAP)
    df["health_concerns"] = df["health_concerns"].map(HEALTH_MAP)

    df["consume_frequency(weekly)"] = (
        df["consume_frequency(weekly)"].map(FREQ_MAP)
    )

    df["preferable_consumption_size"] = (
        df["preferable_consumption_size"].map(SIZE_MAP)
    )

    drop_cols = [
        "respondent_id",
        "price_range",
        "age",
        "cf_num",
        "ab_num",
        "zone_num",
        "income_num",
    ]

    df = df.drop(
        columns=[c for c in drop_cols if c in df.columns]
    )

    return df


def fit_transform_features(df):
    data = df.copy()

    imputer = SimpleImputer(
        strategy="most_frequent"
    )

    data[IMPUTE_COLS] = imputer.fit_transform(
        data[IMPUTE_COLS]
    )

    data = _business_features(data)

    cat_cols = data.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    num_cols = [
        c for c in data.columns
        if c not in cat_cols
    ]

    ohe = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False,
    )

    encoded = ohe.fit_transform(
        data[cat_cols]
    )

    encoded_cols = ohe.get_feature_names_out(
        cat_cols
    )

    encoded_df = pd.DataFrame(
        encoded,
        columns=encoded_cols,
        index=data.index,
    )

    X = pd.concat(
        [data[num_cols], encoded_df],
        axis=1,
    )

    artifacts = {
        "imputer": imputer,
        "ohe": ohe,
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "feature_names": X.columns.tolist(),
    }

    return X, artifacts


def transform_features(df, artifacts):
    data = df.copy()

    data[IMPUTE_COLS] = (
        artifacts["imputer"].transform(
            data[IMPUTE_COLS]
        )
    )

    data = _business_features(data)

    encoded = artifacts["ohe"].transform(
        data[artifacts["cat_cols"]]
    )

    encoded_df = pd.DataFrame(
        encoded,
        columns=artifacts["ohe"].get_feature_names_out(
            artifacts["cat_cols"]
        ),
        index=data.index,
    )

    X = pd.concat(
        [
            data[artifacts["num_cols"]],
            encoded_df
        ],
        axis=1,
    )

    # Guarantee training schema/order
    X = X.reindex(
        columns=artifacts["feature_names"],
        fill_value=0,
    )

    return X

Overwriting /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/preprocessing/feature_utils.py


In [36]:
%%writefile /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/training/requirements.txt


xgboost==2.1.4
joblib>=1.3,<2

Overwriting /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/training/requirements.txt


In [37]:
from pathlib import Path

repo = Path(
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS"
)

(repo / "src/preprocessing/__init__.py").touch()
(repo / "src/training/__init__.py").touch()

In [38]:
%%writefile /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/training/train.py

import glob
import json
import os
import sys

from pathlib import Path
from datetime import datetime, timezone

import joblib
import pandas as pd
import sklearn
import xgboost

from xgboost import XGBClassifier


# --------------------------------------------------
# Make src/ importable inside SageMaker container
# --------------------------------------------------

SRC_DIR = Path(__file__).resolve().parents[1]

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


from preprocessing.feature_utils import (
    PRICE_MAP,
    fit_transform_features,
)


def find_file(directory, extension):
    matches = glob.glob(
        os.path.join(directory, f"*{extension}")
    )

    if not matches:
        raise FileNotFoundError(
            f"No {extension} file found in {directory}"
        )

    return matches[0]


def main():

    training_dir = os.environ["SM_CHANNEL_TRAINING"]
    config_dir = os.environ["SM_CHANNEL_CONFIG"]
    model_dir = os.environ["SM_MODEL_DIR"]

    # --------------------------------------------------
    # Load managed S3 inputs
    # --------------------------------------------------

    data_path = find_file(
        training_dir,
        ".csv"
    )

    params_path = find_file(
        config_dir,
        ".json"
    )

    df = pd.read_csv(data_path)

    with open(params_path, "r") as f:
        best_params = json.load(f)

    print(f"Training rows: {len(df)}")

    # --------------------------------------------------
    # Target
    # --------------------------------------------------

    y = df["price_range"].map(PRICE_MAP)

    if y.isna().any():
        raise ValueError(
            "Unexpected target category detected."
        )

    # --------------------------------------------------
    # Fit production preprocessing on ALL labeled data
    # --------------------------------------------------

    X, preprocessing_artifacts = (
        fit_transform_features(df)
    )

    print(f"Final feature count: {X.shape[1]}")

    if X.isna().sum().sum() != 0:
        raise ValueError(
            "Missing values remain after preprocessing."
        )

    # --------------------------------------------------
    # Frozen Optuna-selected XGBoost configuration
    # --------------------------------------------------

    model = XGBClassifier(
        **best_params,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss",
        verbosity=0,
    )

    model.fit(X, y)

    # --------------------------------------------------
    # Production artifact bundle
    # --------------------------------------------------

    bundle = {
        "model": model,
        "preprocessing": preprocessing_artifacts,
        "price_map": PRICE_MAP,
        "inverse_price_map": {
            v: k for k, v in PRICE_MAP.items()
        },
    }

    os.makedirs(
        model_dir,
        exist_ok=True
    )

    bundle_path = os.path.join(
        model_dir,
        "model_bundle.joblib"
    )

    joblib.dump(
        bundle,
        bundle_path
    )

    # Native XGBoost artifact as an additional portable copy
    model.save_model(
        os.path.join(
            model_dir,
            "xgboost_model.json"
        )
    )

    metadata = {
        "model": "XGBoost",
        "purpose": "production_refit",
        "training_rows": int(len(df)),
        "feature_count": int(X.shape[1]),
        "hyperparameters": best_params,
        "xgboost_version": xgboost.__version__,
        "sklearn_version": sklearn.__version__,
        "trained_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "official_holdout_metrics": {
            "accuracy": 0.9246,
            "macro_f1": 0.9236,
            "ordinal_mae": 0.0754,
        },
    }

    with open(
        os.path.join(
            model_dir,
            "training_metadata.json"
        ),
        "w"
    ) as f:
        json.dump(
            metadata,
            f,
            indent=4
        )

    print("Production model training complete.")
    print(f"Rows used: {len(df)}")
    print(f"Features: {X.shape[1]}")
    print(f"Artifact: {bundle_path}")


if __name__ == "__main__":
    main()

Overwriting /home/sagemaker-user/Beverage-Price-Prediction-AWS/src/training/train.py


In [39]:
import boto3

from sagemaker.train import ModelTrainer
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)

sm_session = Session()

region = sm_session.boto_region_name
role = get_execution_role()

bucket = "krushang-beverage-ml-2026"

TRAIN_INSTANCE_TYPE = "ml.m5.large"

print("Training instance:", TRAIN_INSTANCE_TYPE)

training_data_uri = (
    f"s3://{bucket}/processed/"
    "cleaned_survey_results.csv"
)

params_uri = (
    f"s3://{bucket}/evaluation/"
    "xgboost_best_params.json"
)

model_output_uri = (
    f"s3://{bucket}/models/"
)

print("Region:", region)
print("Training data:", training_data_uri)
print("Parameters:", params_uri)
print("Model output:", model_output_uri)

Training instance: ml.m5.large
Region: ap-south-1
Training data: s3://krushang-beverage-ml-2026/processed/cleaned_survey_results.csv
Parameters: s3://krushang-beverage-ml-2026/evaluation/xgboost_best_params.json
Model output: s3://krushang-beverage-ml-2026/models/


In [40]:
training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2",
    py_version="py3",
    instance_type=TRAIN_INSTANCE_TYPE,
)

print(training_image)

720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3


In [41]:
source_code = SourceCode(
    source_dir=(
        "/home/sagemaker-user/"
        "Beverage-Price-Prediction-AWS/src"
    ),
    entry_script="training/train.py",
    requirements="training/requirements.txt",
)

compute = Compute(
    instance_type=TRAIN_INSTANCE_TYPE,
    instance_count=1,
    volume_size_in_gb=10,
)

stopping_condition = StoppingCondition(
    max_runtime_in_seconds=900
)

trainer = ModelTrainer(
    training_image=training_image,
    source_code=source_code,
    role=role,
    compute=compute,
    stopping_condition=stopping_condition,
    output_data_config=OutputDataConfig(
        s3_output_path=model_output_uri
    ),
    base_job_name="beverage-xgboost-production",
    sagemaker_session=sm_session,
)

[09/04/26 07:40:49] INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=5832438;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=5832439;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=5832444;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=5832445;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-le                     
                             arn:1.4-2-cpu-py3                                                                     

In [42]:
training_job = trainer.train(
    input_data_config=[
        InputData(
            channel_name="training",
            data_source=training_data_uri,
            content_type="text/csv",
        ),
        InputData(
            channel_name="config",
            data_source=params_uri,
            content_type="application/json",
        ),
    ]
)

[09/04/26 07:40:50] INFO     Creating training_job resource.                                     ]8;id=5832450;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5832451;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31238\31238]8;;\

Output()

[09/04/26 07:42:36] INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5832456;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5832457;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Starting training script                                                              

                    INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5832462;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5832463;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             ++ /usr/bin/python3 --version                                                         

                    INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5832468;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5832469;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             ++ echo /opt/ml/input/config/resourceconfig.json:                                     

...

                    INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5833086;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5833087;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Features: 27                                                                          

                    INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5833092;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5833093;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Artifact: /opt/ml/model/model_bundle.joblib                                           

                    INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5833098;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5833099;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     beverage-xgboost-production-20260904074049/algo-1-1788507688:       ]8;id=5833104;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5833105;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Training Container Execution Completed                                                

[09/04/26 07:43:13] INFO     Final Resource Status: Completed                                    ]8;id=5833110;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5833111;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31590\31590]8;;\

In [43]:
import boto3

sm = boto3.client(
    "sagemaker",
    region_name="ap-south-1"
)

job_name = "beverage-xgboost-production-20260904074049"

desc = sm.describe_training_job(
    TrainingJobName=job_name
)

print("Status:", desc["TrainingJobStatus"])
print(
    "Model artifact:",
    desc["ModelArtifacts"]["S3ModelArtifacts"]
)

Status: Completed
Model artifact: s3://krushang-beverage-ml-2026/models/beverage-xgboost-production-20260904074049/output/model.tar.gz


In [44]:
model_artifact_uri = (
    desc["ModelArtifacts"]["S3ModelArtifacts"]
)

print(model_artifact_uri)

s3://krushang-beverage-ml-2026/models/beverage-xgboost-production-20260904074049/output/model.tar.gz


In [45]:
import boto3
import tarfile
from pathlib import Path
from urllib.parse import urlparse

model_artifact_uri = desc["ModelArtifacts"]["S3ModelArtifacts"]

parsed = urlparse(model_artifact_uri)

artifact_bucket = parsed.netloc
artifact_key = parsed.path.lstrip("/")

local_tar = Path("/tmp/beverage_model.tar.gz")

s3 = boto3.client(
    "s3",
    region_name="ap-south-1"
)

# Download SageMaker model artifact
s3.download_file(
    artifact_bucket,
    artifact_key,
    str(local_tar)
)

print("Downloaded to:", local_tar)
print("Size:", local_tar.stat().st_size, "bytes")

# Inspect archive contents
with tarfile.open(local_tar, "r:gz") as tar:
    members = tar.getnames()

print("\nFiles inside model.tar.gz:")

for name in members:
    print(" -", name)

Downloaded to: /tmp/beverage_model.tar.gz
Size: 5077151 bytes

Files inside model.tar.gz:
 - model_bundle.joblib
 - training_metadata.json
 - xgboost_model.json


In [46]:
import json
import tarfile

extract_dir = Path("/tmp/beverage_model")

extract_dir.mkdir(
    parents=True,
    exist_ok=True
)

with tarfile.open(local_tar, "r:gz") as tar:
    tar.extractall(extract_dir)

metadata_path = (
    extract_dir
    / "training_metadata.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=4))

{
    "model": "XGBoost",
    "purpose": "production_refit",
    "training_rows": 29956,
    "feature_count": 27,
    "hyperparameters": {
        "n_estimators": 700,
        "max_depth": 6,
        "learning_rate": 0.12034778187740461,
        "subsample": 0.9350139863013668,
        "colsample_bytree": 0.9552153727216537,
        "min_child_weight": 5,
        "gamma": 0.13876309692419167,
        "reg_alpha": 0.03870751721089646,
        "reg_lambda": 1.9028977942385588
    },
    "xgboost_version": "2.1.4",
    "sklearn_version": "1.4.2",
    "trained_at_utc": "2026-09-04T07:42:52.893333+00:00",
    "official_holdout_metrics": {
        "accuracy": 0.9246,
        "macro_f1": 0.9236,
        "ordinal_mae": 0.0754
    }
}


/tmp/ipykernel_496/3469633564.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


In [47]:
import joblib

bundle_path = (
    extract_dir
    / "model_bundle.joblib"
)

bundle = joblib.load(bundle_path)

print("Bundle keys:", bundle.keys())
print("Model type:", type(bundle["model"]))
print(
    "Preprocessing keys:",
    bundle["preprocessing"].keys()
)

print("Price map:", bundle["price_map"])
print(
    "Inverse price map:",
    bundle["inverse_price_map"]
)

Bundle keys: dict_keys(['model', 'preprocessing', 'price_map', 'inverse_price_map'])
Model type: <class 'xgboost.sklearn.XGBClassifier'>
Preprocessing keys: dict_keys(['imputer', 'ohe', 'cat_cols', 'num_cols', 'feature_names'])
Price map: {'50-100': 0, '100-150': 1, '150-200': 2, '200-250': 3}
Inverse price map: {0: '50-100', 1: '100-150', 2: '150-200', 3: '200-250'}


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


====================================================
NOTEBOOK 04 — SAGEMAKER MANAGED TRAINING
====================================================

Status: COMPLETED + VALIDATED ✅

Instance:
ml.m5.large

Region:
ap-south-1

Input:
s3://krushang-beverage-ml-2026/
processed/cleaned_survey_results.csv

Hyperparameters:
s3://krushang-beverage-ml-2026/
evaluation/xgboost_best_params.json

Training rows:
29,956

Final features:
27

Model:
XGBoost

Final model artifact:
s3://krushang-beverage-ml-2026/models/
beverage-xgboost-production-20260904074049/
output/model.tar.gz

Artifact contents:
- model_bundle.joblib
- xgboost_model.json
- training_metadata.json

Official holdout:
Accuracy = 0.9246
Macro F1 = 0.9236
Ordinal MAE = 0.0754
====================================================m